# MNIST One-Layer ViT Loss Landscape: Raw Weights vs BigVAE Latents

This notebook studies 2D loss landscapes around a trained one-layer ViT on MNIST.

It compares two parameterizations:

1. **Raw weights**: direct perturbations of full model weights without a decoder.
2. **Latents**: perturbations in latent slots decoded by a frozen BigVAE decoder into 2D linear weights.

For each parameterization, it samples `NUM_DIRECTIONS` random 2D directions, applies filter-wise normalization (FN), evaluates `L(alpha,beta)` on a grid, and estimates:

- `S_rho = max_{alpha^2 + beta^2 <= rho^2} (L(alpha,beta) - L(0,0))`
- `M_rho = mean_{alpha^2 + beta^2 <= rho^2} (L(alpha,beta) - L(0,0))` as a discrete integral estimate
- `A_tau,rho = Area{(alpha,beta): alpha^2 + beta^2 <= rho^2, L(alpha,beta) <= L(0,0) + tau}`

Set `BIG_VAE_CHECKPOINT` below to enable the latent-decoder part. If it is empty, only the raw-weight baseline runs.

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import matplotlib.pyplot as plt
from IPython.display import display

try:
    from torchvision import datasets, transforms
except Exception as exc:
    raise RuntimeError('torchvision is required for this notebook') from exc

try:
    from omegaconf import OmegaConf
    from experiments.train_big_vae import _build_model_cfg, _normalize_model_state_dict_keys
    from models.weight_quantile_vae import BigWeightVAE, build_weight_quantile_vae
except Exception as exc:
    OmegaConf = None
    BigWeightVAE = None
    build_weight_quantile_vae = None
    _build_model_cfg = None
    _normalize_model_state_dict_keys = None
    print('BigVAE imports unavailable; raw-weight landscape still works:', repr(exc))

torch.set_float32_matmul_precision('high')
plt.rcParams['figure.dpi'] = 120


In [ ]:
@dataclass
class Config:
    seed: int = 42
    device: str = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    data_dir: str = './data/mnist'
    train_subset: int = 12000
    eval_subset: int = 2048
    landscape_max_batches: int = 4  # 0 means full eval loader for every grid point
    batch_size: int = 256
    eval_batch_size: int = 512
    num_workers: int = 2

    # One-layer ViT for MNIST.
    image_size: int = 28
    patch_size: int = 7
    hidden_dim: int = 128
    num_heads: int = 4
    mlp_ratio: float = 2.0
    num_classes: int = 10

    train_epochs: int = 3
    train_lr: float = 3e-4
    train_weight_decay: float = 0.05

    # Landscape grid.
    num_directions: int = 8
    grid_points: int = 31
    rho_max: float = 1.0
    rho_values: tuple[float, ...] = (0.25, 0.5, 0.75, 1.0)
    tau_values: tuple[float, ...] = (0.01, 0.05, 0.1)

    # BigVAE latent landscape. Leave empty to skip latent path.
    big_vae_checkpoint: str = ''
    big_vae_tile_T_patches: int = 16
    big_vae_tile_d_out: int = 8
    big_vae_latent_init: str = 'random'  # random | base
    latent_base_mode: str = 'encoder'  # encoder | fit | encoder_fit | random
    latent_fit_steps: int = 0
    latent_fit_lr: float = 1e-2
    encoder_context_batches: int = 2
    encoder_context_rows: int = 1024


cfg = Config()
BIG_VAE_CHECKPOINT = cfg.big_vae_checkpoint  # set path here, or leave empty for raw-only
device = torch.device(cfg.device)

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
print(cfg)
print('device =', device)


In [ ]:
def make_mnist_loaders(cfg: Config) -> tuple[DataLoader, DataLoader]:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    train_set = datasets.MNIST(root=cfg.data_dir, train=True, download=True, transform=transform)
    eval_set = datasets.MNIST(root=cfg.data_dir, train=False, download=True, transform=transform)
    if cfg.train_subset > 0:
        train_set = Subset(train_set, list(range(min(cfg.train_subset, len(train_set)))))
    if cfg.eval_subset > 0:
        eval_set = Subset(eval_set, list(range(min(cfg.eval_subset, len(eval_set)))))
    pin = device.type == 'cuda'
    return (
        DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=pin),
        DataLoader(eval_set, batch_size=cfg.eval_batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=pin),
    )


class OneLayerViT(nn.Module):
    def __init__(self, cfg: Config) -> None:
        super().__init__()
        assert cfg.image_size % cfg.patch_size == 0
        assert cfg.hidden_dim % cfg.num_heads == 0
        self.cfg = cfg
        self.num_patches = (cfg.image_size // cfg.patch_size) ** 2
        patch_dim = cfg.patch_size * cfg.patch_size
        self.patch_embed = nn.Linear(patch_dim, cfg.hidden_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, cfg.hidden_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, cfg.hidden_dim))
        self.norm1 = nn.LayerNorm(cfg.hidden_dim)
        self.attn = nn.MultiheadAttention(cfg.hidden_dim, cfg.num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(cfg.hidden_dim)
        mlp_dim = int(round(cfg.hidden_dim * cfg.mlp_ratio))
        self.mlp = nn.Sequential(nn.Linear(cfg.hidden_dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, cfg.hidden_dim))
        self.norm = nn.LayerNorm(cfg.hidden_dim)
        self.head = nn.Linear(cfg.hidden_dim, cfg.num_classes)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.pos_embed, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        p = self.cfg.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p).contiguous()
        x = x.view(B, 1, -1, p, p).squeeze(1).flatten(2).contiguous()
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        h = self.norm1(x)
        attn, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn
        x = x + self.mlp(self.norm2(x))
        x = self.norm(x)
        return self.head(x[:, 0])


@torch.no_grad()
def evaluate_loss(model: nn.Module, loader: DataLoader, max_batches: int | None = None) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch_idx, (x, y) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction='sum')
        total_loss += float(loss.detach().cpu())
        total_correct += int((logits.argmax(dim=-1) == y).sum().detach().cpu())
        total += int(y.numel())
    return total_loss / max(1, total), total_correct / max(1, total)


def train_base_model(cfg: Config, train_loader: DataLoader, eval_loader: DataLoader) -> OneLayerViT:
    model = OneLayerViT(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.train_lr, weight_decay=cfg.train_weight_decay)
    for epoch in range(1, cfg.train_epochs + 1):
        model.train()
        running = 0.0
        seen = 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            running += float(loss.detach().cpu()) * int(y.numel())
            seen += int(y.numel())
        val_loss, val_acc = evaluate_loss(model, eval_loader)
        print(f'epoch={epoch} train_loss={running/max(1,seen):.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
    return model


train_loader, eval_loader = make_mnist_loaders(cfg)
base_model = train_base_model(cfg, train_loader, eval_loader)
base_loss, base_acc = evaluate_loss(base_model, eval_loader)
base_state = {k: v.detach().clone() for k, v in base_model.state_dict().items()}
print('base_loss', base_loss, 'base_acc', base_acc)


In [ ]:
@torch.no_grad()
def collect_linear_input_contexts(model: OneLayerViT, loader: DataLoader, cfg: Config) -> dict[str, torch.Tensor]:
    """Collect real X matrices for each 2D linear weight decoded by BigVAE.

    Returned tensors follow the BigVAE convention X=[n,d_in] for W=[d_in,d_out].
    """
    model.eval()
    contexts: dict[str, list[torch.Tensor]] = {
        'patch_embed.weight': [],
        'attn.in_proj_weight': [],
        'attn.out_proj.weight': [],
        'mlp.0.weight': [],
        'mlp.2.weight': [],
        'head.weight': [],
    }
    p = cfg.patch_size
    for batch_idx, (images, _labels) in enumerate(loader):
        if int(cfg.encoder_context_batches) > 0 and batch_idx >= int(cfg.encoder_context_batches):
            break
        images = images.to(device, non_blocking=True)
        B = int(images.shape[0])
        patches = images.unfold(2, p, p).unfold(3, p, p).contiguous()
        patches = patches.view(B, 1, -1, p, p).squeeze(1).flatten(2).contiguous()
        contexts['patch_embed.weight'].append(patches.reshape(-1, patches.shape[-1]).detach().cpu())

        x = model.patch_embed(patches)
        cls = model.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + model.pos_embed
        h = model.norm1(x)
        contexts['attn.in_proj_weight'].append(h.reshape(-1, h.shape[-1]).detach().cpu())

        qkv = F.linear(h, model.attn.in_proj_weight, model.attn.in_proj_bias)
        qkv = qkv.view(B, h.shape[1], 3, cfg.num_heads, cfg.hidden_dim // cfg.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(dim=0)
        attn_pre_out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=False)
        attn_pre_out = attn_pre_out.transpose(1, 2).contiguous().view(B, h.shape[1], cfg.hidden_dim)
        contexts['attn.out_proj.weight'].append(attn_pre_out.reshape(-1, attn_pre_out.shape[-1]).detach().cpu())

        attn = F.linear(attn_pre_out, model.attn.out_proj.weight, model.attn.out_proj.bias)
        x = x + attn
        mlp_in = model.norm2(x)
        contexts['mlp.0.weight'].append(mlp_in.reshape(-1, mlp_in.shape[-1]).detach().cpu())
        mlp_hidden = F.gelu(model.mlp[0](mlp_in))
        contexts['mlp.2.weight'].append(mlp_hidden.reshape(-1, mlp_hidden.shape[-1]).detach().cpu())
        x = x + model.mlp[2](mlp_hidden)
        x = model.norm(x)
        contexts['head.weight'].append(x[:, 0].detach().cpu())

    output: dict[str, torch.Tensor] = {}
    for name, parts in contexts.items():
        if not parts:
            continue
        tensor = torch.cat(parts, dim=0).to(device=device, dtype=torch.float32)
        if int(cfg.encoder_context_rows) > 0 and int(tensor.shape[0]) > int(cfg.encoder_context_rows):
            tensor = tensor[: int(cfg.encoder_context_rows)].contiguous()
        output[name] = tensor
    return output


encoder_contexts = collect_linear_input_contexts(base_model, train_loader, cfg)
print({name: tuple(x.shape) for name, x in encoder_contexts.items()})


In [ ]:
def clone_state(state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {k: v.detach().clone() for k, v in state.items()}


def is_weight_tensor(name: str, tensor: torch.Tensor) -> bool:
    return tensor.ndim >= 2 and name.endswith('weight')


def filterwise_normalized_direction(state: dict[str, torch.Tensor], *, generator: torch.Generator) -> dict[str, torch.Tensor]:
    direction: dict[str, torch.Tensor] = {}
    for name, value in state.items():
        if not torch.is_floating_point(value) or not is_weight_tensor(name, value):
            direction[name] = torch.zeros_like(value)
            continue
        noise = torch.randn(value.shape, generator=generator, device=value.device, dtype=value.dtype)
        flat_value = value.reshape(value.shape[0], -1)
        flat_noise = noise.reshape(noise.shape[0], -1)
        value_norm = flat_value.norm(dim=1, keepdim=True).clamp_min(1e-12)
        noise_norm = flat_noise.norm(dim=1, keepdim=True).clamp_min(1e-12)
        direction[name] = (flat_noise / noise_norm * value_norm).reshape_as(value)
    return direction


def apply_state_direction(
    model: nn.Module,
    base_state: dict[str, torch.Tensor],
    d1: dict[str, torch.Tensor],
    d2: dict[str, torch.Tensor],
    alpha: float,
    beta: float,
) -> None:
    new_state = {}
    for name, base in base_state.items():
        new_state[name] = base + float(alpha) * d1[name] + float(beta) * d2[name]
    model.load_state_dict(new_state, strict=True)


def make_raw_direction_pairs(state: dict[str, torch.Tensor], num_pairs: int, seed: int) -> list[tuple[dict[str, torch.Tensor], dict[str, torch.Tensor]]]:
    generator = torch.Generator(device=next(iter(state.values())).device)
    generator.manual_seed(seed)
    pairs = []
    for _ in range(num_pairs):
        pairs.append((filterwise_normalized_direction(state, generator=generator), filterwise_normalized_direction(state, generator=generator)))
    return pairs


In [ ]:
@dataclass
class LandscapeResult:
    kind: str
    direction_idx: int
    alphas: np.ndarray
    betas: np.ndarray
    losses: np.ndarray
    base_loss: float


def evaluate_landscape(
    *,
    kind: str,
    direction_pairs: list[Any],
    base_loss: float,
    eval_fn,
    grid_points: int,
    rho_max: float,
) -> list[LandscapeResult]:
    alphas = np.linspace(-rho_max, rho_max, grid_points)
    betas = np.linspace(-rho_max, rho_max, grid_points)
    results = []
    for direction_idx, pair in enumerate(direction_pairs):
        losses = np.full((grid_points, grid_points), np.nan, dtype=np.float64)
        t0 = time.time()
        for i, alpha in enumerate(alphas):
            for j, beta in enumerate(betas):
                losses[i, j] = float(eval_fn(pair, float(alpha), float(beta)))
        results.append(LandscapeResult(kind=kind, direction_idx=direction_idx, alphas=alphas, betas=betas, losses=losses, base_loss=base_loss))
        print(f'{kind} direction={direction_idx} done in {time.time()-t0:.1f}s min={np.nanmin(losses):.4f} max={np.nanmax(losses):.4f}')
    return results


def compute_landscape_metrics(results: list[LandscapeResult], rho_values: Iterable[float], tau_values: Iterable[float]) -> pd.DataFrame:
    rows = []
    for result in results:
        A, B = np.meshgrid(result.alphas, result.betas, indexing='ij')
        delta = result.losses - result.base_loss
        step_alpha = float(abs(result.alphas[1] - result.alphas[0])) if len(result.alphas) > 1 else 1.0
        step_beta = float(abs(result.betas[1] - result.betas[0])) if len(result.betas) > 1 else 1.0
        cell_area = step_alpha * step_beta
        radius2 = A * A + B * B
        for rho in rho_values:
            disk = radius2 <= float(rho) ** 2 + 1e-12
            disk_delta = delta[disk]
            row = {
                'kind': result.kind,
                'direction_idx': result.direction_idx,
                'rho': float(rho),
                'S_rho': float(np.nanmax(disk_delta)),
                'M_rho': float(np.nanmean(disk_delta)),
                'disk_grid_points': int(disk.sum()),
            }
            for tau in tau_values:
                good = disk & (result.losses <= result.base_loss + float(tau))
                row[f'A_tau={tau:g}_rho'] = float(good.sum() * cell_area)
                row[f'A_frac_tau={tau:g}_rho'] = float(good.sum() / max(1, disk.sum()))
            rows.append(row)
    return pd.DataFrame(rows)


def plot_landscape(result: LandscapeResult, title: str | None = None) -> None:
    A, B = np.meshgrid(result.alphas, result.betas, indexing='ij')
    delta = result.losses - result.base_loss
    plt.figure(figsize=(5, 4))
    levels = 30
    contour = plt.contourf(A, B, delta, levels=levels, cmap='viridis')
    plt.colorbar(contour, label='L(alpha,beta) - L(0,0)')
    plt.contour(A, B, delta, colors='black', linewidths=0.35, levels=10, alpha=0.5)
    plt.scatter([0], [0], c='red', s=18)
    plt.xlabel('alpha')
    plt.ylabel('beta')
    plt.title(title or f'{result.kind} direction {result.direction_idx}')
    plt.tight_layout()
    plt.show()


def plot_metric_boxplots(metrics: pd.DataFrame, metric: str) -> None:
    plt.figure(figsize=(6, 4))
    labels = []
    values = []
    for (kind, rho), group in metrics.groupby(['kind', 'rho']):
        labels.append(f'{kind}\nrho={rho:g}')
        values.append(group[metric].to_numpy())
    plt.boxplot(values, labels=labels, showmeans=True)
    plt.ylabel(metric)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
raw_model = OneLayerViT(cfg).to(device)
raw_model.load_state_dict(base_state, strict=True)
raw_direction_pairs = make_raw_direction_pairs(base_state, cfg.num_directions, cfg.seed + 1000)

def raw_eval_fn(pair, alpha: float, beta: float) -> float:
    d1, d2 = pair
    apply_state_direction(raw_model, base_state, d1, d2, alpha, beta)
    max_batches = None if cfg.landscape_max_batches <= 0 else cfg.landscape_max_batches
    loss, _ = evaluate_loss(raw_model, eval_loader, max_batches=max_batches)
    return loss

raw_results = evaluate_landscape(
    kind='raw',
    direction_pairs=raw_direction_pairs,
    base_loss=base_loss,
    eval_fn=raw_eval_fn,
    grid_points=cfg.grid_points,
    rho_max=cfg.rho_max,
)
raw_metrics = compute_landscape_metrics(raw_results, cfg.rho_values, cfg.tau_values)
display(raw_metrics.head())
plot_landscape(raw_results[0], 'Raw-weight FN landscape: direction 0')


## BigVAE latent-induced landscape

This section is optional. It loads a frozen BigVAE decoder and builds a latent parameterization of the 2D linear weights in the MNIST ViT. By default, the base latent point is produced by the BigVAE encoder from trained weights and real layer input activations `X`. Optional MSE refinement is available via `latent_base_mode='encoder_fit'` or `latent_base_mode='fit'`. Then random latent directions are sampled with slot-wise normalization, the latent slots are perturbed by `(alpha,beta)`, decoded to raw model weights, and the MNIST loss is evaluated.

The decoder sees fixed stage-like tiles: `d_in = patch_size_bigvae * big_vae_tile_T_patches`, `d_out = big_vae_tile_d_out`. Small matrices are packed by patch rows into these tiles to avoid paying one latent for mostly empty padding.

In [ ]:
@dataclass
class MatrixSpec:
    original_shape: tuple[int, ...]
    matrix_shape: tuple[int, int]
    transposed: bool


@dataclass
class TileSegment:
    name: str
    tile_row_start: int
    row_start: int
    row_len: int
    col_start: int
    col_len: int


@dataclass
class DecodeTile:
    key: str
    segments: list[TileSegment]


def tensor_to_matrix(tensor: torch.Tensor, spec: MatrixSpec) -> torch.Tensor:
    matrix = tensor.reshape(int(spec.original_shape[0]), -1)
    if spec.transposed:
        matrix = matrix.transpose(0, 1)
    return matrix.contiguous()


def matrix_to_tensor(matrix: torch.Tensor, spec: MatrixSpec) -> torch.Tensor:
    restored = matrix.transpose(0, 1) if spec.transposed else matrix
    return restored.contiguous().reshape(spec.original_shape)


def load_frozen_big_vae(checkpoint_path: str, device: torch.device):
    if not checkpoint_path:
        return None
    if BigWeightVAE is None:
        raise RuntimeError('BigVAE imports are unavailable in this kernel')
    path = Path(checkpoint_path).expanduser()
    payload = torch.load(path, map_location='cpu', weights_only=False)
    cfg_payload = OmegaConf.create(payload['config'])
    model_cfg = _build_model_cfg(cfg_payload)
    model = build_weight_quantile_vae(model_cfg)
    state = payload.get('model_state', payload.get('state_dict'))
    missing, unexpected = model.load_state_dict(_normalize_model_state_dict_keys(state), strict=False)
    if missing or unexpected:
        raise RuntimeError(f'BigVAE state mismatch: missing={missing[:8]} unexpected={unexpected[:8]}')
    model.to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model


class BigVAELatentAdapter(nn.Module):
    def __init__(self, big_vae: nn.Module, base_state: dict[str, torch.Tensor], cfg: Config) -> None:
        super().__init__()
        self.big_vae = big_vae
        self.patch_size = int(big_vae.cfg.patch_size)
        self.tile_T = int(cfg.big_vae_tile_T_patches)
        self.tile_d_in = self.patch_size * self.tile_T
        self.tile_d_out = int(cfg.big_vae_tile_d_out)
        self.use_dist = bool(getattr(big_vae, 'use_distribution_encoder', False))
        self.d_dist = int(big_vae.cfg.distribution.d_dist)
        self.specs: dict[str, MatrixSpec] = {}
        self.tiles: list[DecodeTile] = []
        self.direct_state = {k: v.detach().clone().to(device) for k, v in base_state.items()}
        self.latents = nn.ParameterDict()
        base_latent = big_vae.latent_base.detach().clone().to(device)
        for name, value in base_state.items():
            if not (value.ndim == 2 and name.endswith('weight')):
                continue
            spec = MatrixSpec(tuple(value.shape), (int(value.shape[1]), int(value.shape[0])), True)
            self.specs[name] = spec
            rows, cols = spec.matrix_shape
            pending: list[TileSegment] = []
            current_rows = 0
            tile_idx = 0
            def flush() -> None:
                nonlocal pending, tile_idx, current_rows
                if not pending:
                    return
                key = f't{len(self.tiles):05d}'
                self.tiles.append(DecodeTile(key=key, segments=list(pending)))
                if cfg.big_vae_latent_init == 'base':
                    latent = base_latent.clone()
                else:
                    latent = torch.randn_like(base_latent) * 0.02
                self.latents[key] = nn.Parameter(latent)
                pending = []
                current_rows = 0
                tile_idx += 1
            for col_start in range(0, cols, self.tile_d_out):
                col_len = min(self.tile_d_out, cols - col_start)
                for row_start in range(0, rows, self.patch_size):
                    row_len = min(self.patch_size, rows - row_start)
                    if current_rows > 0 and current_rows + row_len > self.tile_d_in:
                        flush()
                    pending.append(TileSegment(name, current_rows, row_start, row_len, col_start, col_len))
                    current_rows += row_len
                    if current_rows >= self.tile_d_in:
                        flush()
            flush()

    def latent_numel(self) -> int:
        return sum(p.numel() for p in self.latents.values())

    def decoded_numel(self) -> int:
        return sum(int(np.prod(self.direct_state[name].shape)) for name in self.specs)

    def decode_matrices(self, latent_override: dict[str, torch.Tensor] | None = None) -> dict[str, torch.Tensor]:
        result = {name: torch.zeros(spec.matrix_shape, device=device) for name, spec in self.specs.items()}
        if not self.tiles:
            return result
        keys = [tile.key for tile in self.tiles]
        latents = torch.stack([(latent_override or self.latents)[key] for key in keys], dim=0)
        B = latents.shape[0]
        patch_mask = torch.zeros(B, self.tile_T, device=device, dtype=torch.bool)
        d_in_mask = torch.zeros(B, self.tile_d_in, device=device, dtype=torch.bool)
        d_out_mask = torch.zeros(B, self.tile_d_out, device=device, dtype=torch.bool)
        for i, tile in enumerate(self.tiles):
            used_rows = 0
            used_cols = 0
            for seg in tile.segments:
                row_end = seg.tile_row_start + seg.row_len
                d_in_mask[i, seg.tile_row_start:row_end] = True
                used_rows = max(used_rows, row_end)
                used_cols = max(used_cols, seg.col_len)
            patch_mask[i, :math.ceil(used_rows / self.patch_size)] = True
            d_out_mask[i, :used_cols] = True
        dist_patch = torch.zeros(B, self.tile_T, self.d_dist, device=device, dtype=latents.dtype) if self.use_dist else None
        decoded = self.big_vae._decode_from_latent_slots(
            latents,
            dist_patch_by_patch=dist_patch,
            patch_mask=patch_mask,
            d_in_mask=d_in_mask,
            d_out_mask=d_out_mask,
            d_in=self.tile_d_in,
            d_out=self.tile_d_out,
            d_in_pad=self.tile_d_in,
            T=self.tile_T,
        )[0]
        for i, tile in enumerate(self.tiles):
            for seg in tile.segments:
                row_end = seg.tile_row_start + seg.row_len
                result[seg.name][seg.row_start:seg.row_start+seg.row_len, seg.col_start:seg.col_start+seg.col_len] = decoded[i, seg.tile_row_start:row_end, :seg.col_len]
        return result

    def tile_weight_and_context(
        self,
        tile: DecodeTile,
        base_state: dict[str, torch.Tensor],
        contexts: dict[str, torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        first_context = contexts[tile.segments[0].name]
        n = int(first_context.shape[0])
        W_tile = torch.zeros(self.tile_d_in, self.tile_d_out, device=device, dtype=torch.float32)
        X_tile = torch.zeros(n, self.tile_d_in, device=device, dtype=torch.float32)
        d_in_mask = torch.zeros(self.tile_d_in, device=device, dtype=torch.bool)
        d_out_mask = torch.zeros(self.tile_d_out, device=device, dtype=torch.bool)
        for seg in tile.segments:
            if seg.name not in contexts:
                raise KeyError(f'missing encoder context for {seg.name}')
            X_src = contexts[seg.name]
            if int(X_src.shape[0]) != n:
                X_src = X_src[:n]
            matrix = tensor_to_matrix(base_state[seg.name].to(device), self.specs[seg.name])
            row_end = seg.tile_row_start + seg.row_len
            W_tile[seg.tile_row_start:row_end, :seg.col_len] = matrix[
                seg.row_start:seg.row_start + seg.row_len,
                seg.col_start:seg.col_start + seg.col_len,
            ]
            X_tile[:, seg.tile_row_start:row_end] = X_src[:, seg.row_start:seg.row_start + seg.row_len]
            d_in_mask[seg.tile_row_start:row_end] = True
            d_out_mask[:seg.col_len] = True
        return W_tile, X_tile, d_in_mask, d_out_mask

    @torch.no_grad()
    def initialize_latents_from_encoder(self, base_state: dict[str, torch.Tensor], contexts: dict[str, torch.Tensor]) -> None:
        encoded: dict[str, torch.Tensor] = {}
        for idx, tile in enumerate(self.tiles):
            W_tile, X_tile, d_in_mask, d_out_mask = self.tile_weight_and_context(tile, base_state, contexts)
            W_b = W_tile.unsqueeze(0)
            X_b = X_tile.unsqueeze(0)
            d_in_mask_b = d_in_mask.unsqueeze(0)
            d_out_mask_b = d_out_mask.unsqueeze(0)
            T, d_in_pad, patch_mask, _structural_patch_mask, dist_var_by_patch, dist_patch_by_patch, dist_var_pooled = self.big_vae._encode_distribution_context(
                X_b,
                x_mask=None,
                d_in_mask=d_in_mask_b,
            )
            latents = self.big_vae._encode_latent_slots(
                W_b,
                T=T,
                d_in_pad=d_in_pad,
                patch_mask=patch_mask,
                d_out_mask=d_out_mask_b,
                dist_var_by_patch=dist_var_by_patch,
                dist_patch_by_patch=dist_patch_by_patch,
                dist_var_pooled=dist_var_pooled,
                return_debug_info=False,
            )
            encoded[tile.key] = latents.squeeze(0).detach().clone()
            if idx == 0 or (idx + 1) % 50 == 0 or idx + 1 == len(self.tiles):
                print(f'encoded latent tile {idx + 1}/{len(self.tiles)}', flush=True)
        for key, value in encoded.items():
            if tuple(self.latents[key].shape) != tuple(value.shape):
                raise RuntimeError(f'encoder latent shape mismatch for {key}: param={tuple(self.latents[key].shape)} encoded={tuple(value.shape)}')
            self.latents[key].copy_(value)

    def materialize_state(self, latent_override: dict[str, torch.Tensor] | None = None) -> dict[str, torch.Tensor]:
        state = {k: v.detach().clone() for k, v in self.direct_state.items()}
        matrices = self.decode_matrices(latent_override)
        for name, matrix in matrices.items():
            state[name] = matrix_to_tensor(matrix, self.specs[name])
        return state


def fit_latents_to_base(adapter: BigVAELatentAdapter, base_state: dict[str, torch.Tensor], steps: int, lr: float) -> None:
    opt = torch.optim.AdamW(adapter.latents.parameters(), lr=lr, weight_decay=0.0)
    targets = {name: tensor_to_matrix(base_state[name].to(device), spec) for name, spec in adapter.specs.items()}
    total = sum(t.numel() for t in targets.values())
    for step in range(1, steps + 1):
        opt.zero_grad(set_to_none=True)
        decoded = adapter.decode_matrices()
        loss = sum(F.mse_loss(decoded[name], target, reduction='sum') for name, target in targets.items()) / max(1, total)
        loss.backward()
        opt.step()
        if step == 1 or step % max(1, steps // 5) == 0 or step == steps:
            print(f'latent fit step={step}/{steps} mse={float(loss.detach().cpu()):.6e}')


def make_latent_direction(adapter: BigVAELatentAdapter, generator: torch.Generator) -> dict[str, torch.Tensor]:
    direction = {}
    for key, latent in adapter.latents.items():
        noise = torch.randn(latent.shape, generator=generator, device=latent.device, dtype=latent.dtype)
        direction[key] = noise / noise.norm().clamp_min(1e-12) * latent.detach().norm().clamp_min(1e-12)
    return direction


def make_latent_direction_pairs(adapter: BigVAELatentAdapter, num_pairs: int, seed: int):
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)
    return [(make_latent_direction(adapter, generator), make_latent_direction(adapter, generator)) for _ in range(num_pairs)]


In [ ]:
latent_results = []
latent_metrics = pd.DataFrame()

if not BIG_VAE_CHECKPOINT:
    print('BIG_VAE_CHECKPOINT is empty; skipping latent-induced landscape.')
else:
    big_vae = load_frozen_big_vae(BIG_VAE_CHECKPOINT, device)
    adapter = BigVAELatentAdapter(big_vae, base_state, cfg).to(device)
    print('latent tiles:', len(adapter.tiles))
    print('decoded linear params:', adapter.decoded_numel())
    print('latent params:', adapter.latent_numel())
    print('effective decoded/latent:', adapter.decoded_numel() / max(1, adapter.latent_numel()))
    mode = str(cfg.latent_base_mode).strip().lower()
    if mode not in {'encoder', 'fit', 'encoder_fit', 'random'}:
        raise ValueError(f'unsupported latent_base_mode={cfg.latent_base_mode!r}')
    if mode in {'encoder', 'encoder_fit'}:
        print('initializing latent base point from BigVAE encoder')
        adapter.initialize_latents_from_encoder(base_state, encoder_contexts)
    if mode in {'fit', 'encoder_fit'} and int(cfg.latent_fit_steps) > 0:
        print('refining latent base point by decoded-weight MSE')
        fit_latents_to_base(adapter, base_state, cfg.latent_fit_steps, cfg.latent_fit_lr)
    latent_base_state = adapter.materialize_state()
    latent_model = OneLayerViT(cfg).to(device)
    latent_model.load_state_dict(latent_base_state, strict=True)
    latent_base_loss, latent_base_acc = evaluate_loss(latent_model, eval_loader)
    print('latent_base_loss', latent_base_loss, 'latent_base_acc', latent_base_acc)
    latent_pairs = make_latent_direction_pairs(adapter, cfg.num_directions, cfg.seed + 2000)

    def latent_eval_fn(pair, alpha: float, beta: float) -> float:
        d1, d2 = pair
        override = {key: adapter.latents[key] + float(alpha) * d1[key] + float(beta) * d2[key] for key in adapter.latents.keys()}
        state = adapter.materialize_state(override)
        latent_model.load_state_dict(state, strict=True)
        max_batches = None if cfg.landscape_max_batches <= 0 else cfg.landscape_max_batches
        loss, _ = evaluate_loss(latent_model, eval_loader, max_batches=max_batches)
        return loss

    latent_results = evaluate_landscape(
        kind='latent',
        direction_pairs=latent_pairs,
        base_loss=latent_base_loss,
        eval_fn=latent_eval_fn,
        grid_points=cfg.grid_points,
        rho_max=cfg.rho_max,
    )
    latent_metrics = compute_landscape_metrics(latent_results, cfg.rho_values, cfg.tau_values)
    display(latent_metrics.head())
    plot_landscape(latent_results[0], 'BigVAE latent-induced landscape: direction 0')


In [ ]:
all_metrics = pd.concat([raw_metrics, latent_metrics], ignore_index=True) if len(latent_metrics) else raw_metrics.copy()
display(all_metrics)

plot_metric_boxplots(all_metrics, 'S_rho')
plot_metric_boxplots(all_metrics, 'M_rho')
for tau in cfg.tau_values:
    col = f'A_frac_tau={tau:g}_rho'
    if col in all_metrics.columns:
        plot_metric_boxplots(all_metrics, col)

out_dir = Path('./artifacts/loss_landscape_mnist_one_layer_vit')
out_dir.mkdir(parents=True, exist_ok=True)
all_metrics.to_csv(out_dir / 'landscape_metrics.csv', index=False)
print('saved metrics to', out_dir / 'landscape_metrics.csv')
